In [1]:
print('Hello')

Hello


In [1]:
import torch

# Check if CUDA is available
cuda_available = torch.cuda.is_available()
print(f"Is GPU available? {cuda_available}")

if cuda_available:
    # Print the name of your local GPU
    print(f"GPU Device Name: {torch.cuda.get_device_name(0)}")
    
    # Target the GPU device explicitly
    device = torch.device("cuda")
    
    # Allocate a tensor directly onto the GPU
    x = torch.rand(3, 3, device=device)
    print("Successfully allocated tensor on GPU:\n", x)

Is GPU available? True
GPU Device Name: NVIDIA GeForce RTX 5070 Ti Laptop GPU
Successfully allocated tensor on GPU:
 tensor([[0.4915, 0.7577, 0.1812],
        [0.3286, 0.4795, 0.3629],
        [0.7004, 0.4549, 0.6024]], device='cuda:0')


In [4]:
pip install load_dataset

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement load_dataset (from versions: none)
ERROR: No matching distribution found for load_dataset


In [1]:
##################[THIS CELL AND THE CELL BELOW ARE AI GENERATED]##################

import torch
from datasets import load_dataset
from transformer_lens import HookedTransformer
from torch.utils.data import DataLoader

# 1. Setup Model (Using GPU if available)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = HookedTransformer.from_pretrained("gpt2-small", device=device)
NUM_NEURONS = model.cfg.d_mlp  # 3072

# 2. Initialize Parallel Trackers
K = 25
# Global tensor to store the highest 25 activation values for each neuron [3072, 25]
top_scores = torch.full((NUM_NEURONS, K), -999.0, device=device)

# Global list matrix to store the tokenized prefixes [3072, 25]
top_prefixes = [[None] * K for _ in range(NUM_NEURONS)]

# 3. Load and Filter Dataset
# Change "wikitext" to the fully qualified "Salesforce/wikitext"
dataset = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")

dataset = dataset.filter(lambda x: len(x["text"].strip()) > 30)
dataloader = DataLoader(dataset, batch_size=8, shuffle=False)

print(f"Processing dataset using vectorized tracking on {device.upper()}...")

# 4. Processing Loop
for batch_idx, batch in enumerate(dataloader):
    text_list = batch["text"]

    # Tokenize [batch_size, seq_len]
    tokens = model.to_tokens(text_list, prepend_bos=True).to(device)
    batch_size, seq_len = tokens.shape

    with torch.no_grad():
        _, cache = model.run_with_cache(
            tokens,
            names_filter=lambda name: name == "blocks.1.mlp.hook_post"
        )

    # Activations shape: [batch_size, seq_len, 3072] -> Permute/Flatten to [3072, batch_size * seq_len]
    flat_acts = cache["blocks.1.mlp.hook_post"].permute(2, 0, 1).reshape(NUM_NEURONS, -1)

    # --- VECTORIZED MATRIX STEP ---
    # Concatenate the new activations to our existing top scores across dim 1
    # Shape: [3072, 25 + (batch_size * seq_len)]
    combined_scores = torch.cat([top_scores, flat_acts], dim=1)

    # Compute Top-K for all 3072 neurons simultaneously on the GPU GPU GPU!
    new_top_vals, new_top_indices = torch.topk(combined_scores, k=K, dim=1, largest=True)

    # Update global scores
    top_scores = new_top_vals

    # Convert top indices to CPU list matrix for history tracking
    indices_cpu = new_top_indices.cpu().tolist()
    tokens_cpu = tokens.cpu()  # Move tokens to CPU for list slicing

    # 5. Fast Token History Slicing (CPU Side)
    for neuron_id in range(NUM_NEURONS):
        updated_neuron_prefixes = []
        neuron_indices = indices_cpu[neuron_id]

        for flat_idx in neuron_indices:
            # If index < K, it references a historically cached prefix
            if flat_idx < K:
                updated_neuron_prefixes.append(top_prefixes[neuron_id][flat_idx])
            else:
                # Map the flat batch index back to (batch_item, sequence_position)
                batch_token_idx = flat_idx - K
                b_item = batch_token_idx // seq_len
                s_pos = batch_token_idx % seq_len

                # Capture everything from the start of the sequence up to the active token
                token_history = tokens_cpu[b_item, :s_pos + 1].tolist()
                updated_neuron_prefixes.append(token_history)

        top_prefixes[neuron_id] = updated_neuron_prefixes

    # Clean up loop elements from GPU VRAM
    del cache, flat_acts, combined_scores, new_top_vals, new_top_indices, tokens
    if device == "cuda":
        torch.cuda.empty_cache()

    if (batch_idx + 1) % 25 == 0:
        print(f"Processed {(batch_idx + 1) * 8} text documents...")

print("\nData collection complete!")

c:\Users\theep\anaconda3\envs\ai-training\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\theep\anaconda3\envs\ai-training\lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\theep\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer 

Loaded pretrained model gpt2-small into HookedTransformer


c:\Users\theep\anaconda3\envs\ai-training\lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\theep\.cache\huggingface\hub\datasets--Salesforce--wikitext. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Filter: 100%|██████████| 4358/4358 [00:00<00:00, 160770.28 examples/s]


Processing dataset using vectorized tracking on CUDA...
Processed 200 text documents...
Processed 400 text documents...
Processed 600 text documents...
Processed 800 text documents...
Processed 1000 text documents...
Processed 1200 text documents...
Processed 1400 text documents...
Processed 1600 text documents...
Processed 1800 text documents...
Processed 2000 text documents...
Processed 2200 text documents...

Data collection complete!


In [8]:
target_neuron = 2660  # Change this to any index up to 3071

print(f"=== Top 5 Activation Contexts for Neuron #{target_neuron} ===\n")
for rank in range(25):
    token_ids = top_prefixes[target_neuron][rank]
    score = top_scores[target_neuron][rank].item()

    if token_ids is None or score == -999.0:
        continue

    # Translate token IDs back to a readable string context
    decoded_context = model.to_string(token_ids)

    # Slice the final token out to visually highlight the exact word that triggered it
    trigger_token_str = model.to_string([token_ids[-1]])

    print(f"[Rank {rank+1}] Activation Score: {score:.4f}")
    # Truncate string for cleaner printing if it's exceptionally long
    print(f"Prefix Window: \"... {decoded_context[-120:]} <-\"")
    print(f"Trigger Token: \"{trigger_token_str.strip()}\"")
    print("-" * 40)

=== Top 5 Activation Contexts for Neuron #2660 ===

[Rank 1] Activation Score: 3.2898
Prefix Window: "... mation of a special commission of economy to drastically reduce expenditures . The most superfluous religious sacrifices <-"
Trigger Token: "sacrifices"
----------------------------------------
[Rank 2] Activation Score: 3.0889
Prefix Window: "... 1 @.@ 5 metres ( 4 @.@ 9 ft ) high . The shape of the stone has been compared both to that of a sarcophagus and an altar <-"
Trigger Token: "altar"
----------------------------------------
[Rank 3] Activation Score: 2.9931
Prefix Window: "... ntury the first Eastern European Jews settled there , and by the 1920s Eugene 's Jewish community began gathering prayer <-"
Trigger Token: "prayer"
----------------------------------------
[Rank 4] Activation Score: 2.9693
Prefix Window: "... tyles have been used in efforts to translate Du Fu 's work into English . As Burton Watson remarks in The Selected Poems <-"
Trigger Token: "ems"
-------------